# User study on ProtoSSL vs ProtoECGNet

* compare ProtoSSL-PILA (no fine-tuning) vs ProtoECGNet on PTB-XL full set with 14 prototypes-per-label
* use samples/labels from PTB-XL (ECG interpretation task) **test set**
    * labels: AMI, RBBB, LBBB, PVC
    * samples: 5 random positive cases per label where both models predict the label correctly
* **case prototype interpretation**: for each sample, pull prototype of that label from model which is maximally similar
    * get similarity score
    * get window of case which resulted in that similarity
* **global prototype interpretation**: for those prototypes, find the window of the prototype and highlight on source ECG


In [1]:
from pathlib import Path

import torch
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm
from sklearn.metrics import roc_curve
from sklearn.preprocessing import StandardScaler
from scipy.signal import butter, filtfilt

from protossl.datasets import PtbxlECGDataset
from protossl.defines import PTBXL_CLIPPED_MEANS, PTBXL_CLIPPED_STDS, PTBXL_TARGETS

ptbxl_path = "/opt/gpudata/ecg/ptb-xl"
labels = ["AMI", "CRBBB", "CLBBB", "PVC"]
cases_per_label = 5 # CPL
protossl = Path("/opt/gpu_working/steven/protossl-outputs/runs-ptbxl/protossl-heedb-pila")
protoecgnet = Path("/opt/gpu_working/steven/protossl-outputs/runs-ptbxl/labsup-proto-direct")
prototypes_per_label = 14 # the PPL used in above models

rng = np.random.default_rng(12)

_ = plt.ioff()

In [2]:
assert all([l in PTBXL_TARGETS for l in labels])
test_ds = PtbxlECGDataset(dataset_path=ptbxl_path, split="test", sampling_rate=100)
train_ds = PtbxlECGDataset(dataset_path=ptbxl_path, split="train", sampling_rate=100)
ecg_id_to_idx = {int(ecg_id): i for i, ecg_id in enumerate(test_ds.sample_ids)}
train_ecg_id_to_idx = {int(ecg_id): i for i, ecg_id in enumerate(train_ds.sample_ids)} # for projection samples

=================load_cached_data==================
Dataset parameters: (/opt/gpudata/ecg/ptb-xl, test, 100Hz)
Cached dataset loaded from /home/songs1/.cache/pass_pclr_cache/9a71164f.pt
=================load_cached_data==================
Dataset parameters: (/opt/gpudata/ecg/ptb-xl, train, 100Hz)
Cached dataset loaded from /home/songs1/.cache/pass_pclr_cache/52f12d62.pt


In [3]:
# ensure that the ECG IDs are not duplicated between sets
assert len(set(ecg_id_to_idx) | set(train_ecg_id_to_idx)) == (len(ecg_id_to_idx) + len(train_ecg_id_to_idx))

In [4]:
label_cases = dict()
Y: torch.Tensor = test_ds.labels # type: ignore
Y_bool = Y == 1
test_idxs = np.arange(len(test_ds))

# have been iteratively selecting based off criteria and manual review, save fixed ECG IDs here
# manual review for unclear morphology in the test case, not in the model prototypes
fixed_cases = {
    "AMI":   [3875,  11319, None,  12235, 20313],
    "CRBBB": [8712,  20252, 19134, None,  None],
    "CLBBB": [18916, 21181, None,  21093, None],
    "PVC":   [6180,  14384, 12576, 10240, 16994],
}

# these ECG IDs should not be used after manual verification
exclude_cases = {
    "AMI": [1014, 16427, 15871],
    "CRBBB": [10359, 15203, 1424, 13855, 14192, 11500],
    "CLBBB": [3479, 18499, 15798, 4432],
    "PVC": [21085, 1157, 16782],
}

protossl_sims = np.load(protossl / "compute-embeddings/latest/test_embeds.npy")
protossl_train_sims = np.load(protossl / "compute-embeddings/latest/train_embeds.npy")
protossl_model = joblib.load(protossl / "model.joblib")
protossl_probs = protossl_model.predict_proba(StandardScaler().fit(protossl_train_sims).transform(protossl_sims))

protoecgnet_sims = np.load(protoecgnet / "compute-embeddings/latest/test_embeds.npy")
protoecgnet_train_sims = np.load(protoecgnet / "compute-embeddings/latest/train_embeds.npy")
protoecgnet_model = joblib.load(protoecgnet / "model.joblib")
protoecgnet_probs = protoecgnet_model.predict_proba(StandardScaler().fit(protoecgnet_train_sims).transform(protoecgnet_sims))

def binarize_predictions(probs, label_idx):
    # binarize youden's J
    fpr, tpr, thresholds = roc_curve(Y[:, label_idx], probs[label_idx][:, 1])
    threshold = thresholds[np.argmax(tpr - fpr)]
    preds = probs[label_idx][:, 1] > threshold
    return preds # return as bool


label_masked_criteria = {
    "AMI": np.logical_and.reduce([
        Y_bool[:, PTBXL_TARGETS.index("AMI")],
        ~(Y_bool[:, PTBXL_TARGETS.index("ILMI")]),
        ~(Y_bool[:, PTBXL_TARGETS.index("IPLMI")]),
        ~(Y_bool[:, PTBXL_TARGETS.index("IPMI")]),
        ~(Y_bool[:, PTBXL_TARGETS.index("PMI")]),
        ~(Y_bool[:, PTBXL_TARGETS.index("IMI")]),
    ]),
    "CRBBB": np.logical_and.reduce([
        Y_bool[:, PTBXL_TARGETS.index("CRBBB")],
        ~(Y_bool[:, PTBXL_TARGETS.index("IRBBB")]),
        ~(Y_bool[:, PTBXL_TARGETS.index("CLBBB")]),
        ~(Y_bool[:, PTBXL_TARGETS.index("ILBBB")]),
        ~(Y_bool[:, PTBXL_TARGETS.index("IVCD")]),
    ]),
    "CLBBB": np.logical_and.reduce([
        Y_bool[:, PTBXL_TARGETS.index("CLBBB")],
        ~(Y_bool[:, PTBXL_TARGETS.index("ILBBB")]),
        ~(Y_bool[:, PTBXL_TARGETS.index("CRBBB")]),
        ~(Y_bool[:, PTBXL_TARGETS.index("IRBBB")]),
        ~(Y_bool[:, PTBXL_TARGETS.index("IVCD")]),
    ]),
    "PVC": np.logical_and.reduce([
        Y_bool[:, PTBXL_TARGETS.index("PVC")],
        ~(Y_bool[:, PTBXL_TARGETS.index("AMI")]),
        ~(Y_bool[:, PTBXL_TARGETS.index("ASMI")]),
        ~(Y_bool[:, PTBXL_TARGETS.index("ALMI")]),
        ~(Y_bool[:, PTBXL_TARGETS.index("ILMI")]),
        ~(Y_bool[:, PTBXL_TARGETS.index("IPLMI")]),
        ~(Y_bool[:, PTBXL_TARGETS.index("IPMI")]),
        ~(Y_bool[:, PTBXL_TARGETS.index("PMI")]),
        ~(Y_bool[:, PTBXL_TARGETS.index("IMI")]),
        ~(Y_bool[:, PTBXL_TARGETS.index("CRBBB")]),
        ~(Y_bool[:, PTBXL_TARGETS.index("IRBBB")]),
        ~(Y_bool[:, PTBXL_TARGETS.index("CLBBB")]),
        ~(Y_bool[:, PTBXL_TARGETS.index("ILBBB")]),
        ~(Y_bool[:, PTBXL_TARGETS.index("IVCD")]),
        ~(Y_bool[:, PTBXL_TARGETS.index("PACE")]),
        ~(Y_bool[:, PTBXL_TARGETS.index("BIGU")]),
        ~(Y_bool[:, PTBXL_TARGETS.index("TRIGU")]),
    ]),
}


for l in labels:
    print("=" * 60)
    label_idx = PTBXL_TARGETS.index(l)

    # can't select cases with only given label, can check with below
    # not_label_idxs = [i for i in range(len(PTBXL_TARGETS)) if i != label_idx]
    # print((test_ds.labels[:, not_label_idxs].sum(axis=1) == 0).sum())

    # case must selectively have label
    has_label_mask = label_masked_criteria[l]
    print(f"Ground-Truth-Selective-{l}: {has_label_mask.sum()}")

    # models must make positively predict label
    protossl_pred_correct_mask = binarize_predictions(protossl_probs, label_idx)
    protoecgnet_pred_correct_mask = binarize_predictions(protoecgnet_probs, label_idx)
    print(f"ProtoSSL-{l}: {protossl_pred_correct_mask.sum()}")
    print(f"ProtoECGNet-{l}: {protoecgnet_pred_correct_mask.sum()}")

    # case must not be in these ECG IDs
    excluded_ecg_ids = exclude_cases[l]
    not_excluded_mask = np.asarray([x not in excluded_ecg_ids for x in test_ds.sample_ids])

    # AND of these masks says that all models have correctly predicted the positive cases
    mask = np.logical_and.reduce([has_label_mask, not_excluded_mask, protossl_pred_correct_mask, protoecgnet_pred_correct_mask])
    print(f"AND-Criteria-{l}: {mask.sum()}")

    # cands/cases are dataset indices (not ECG IDs)
    cands = test_idxs[mask]
    assert len(cands) > cases_per_label
    cases = rng.choice(cands, size=cases_per_label, replace=False)

    # put in fixed cases
    cand_ecg_ids = test_ds.sample_ids[mask]
    for i, _case_ecg_id in enumerate(fixed_cases[l]):
        if _case_ecg_id is None:
            continue
        assert _case_ecg_id in cand_ecg_ids
        cases[i] = ecg_id_to_idx[_case_ecg_id]

    label_cases[l] = cases

# make sure we select unique cases
assert len(set([v for vs in label_cases.values() for v in vs])) == len(labels) * cases_per_label

Ground-Truth-Selective-AMI: 26
ProtoSSL-AMI: 646
ProtoECGNet-AMI: 692
AND-Criteria-AMI: 14
Ground-Truth-Selective-CRBBB: 54
ProtoSSL-CRBBB: 74
ProtoECGNet-CRBBB: 88
AND-Criteria-CRBBB: 47
Ground-Truth-Selective-CLBBB: 54
ProtoSSL-CLBBB: 91
ProtoECGNet-CLBBB: 91
AND-Criteria-CLBBB: 47
Ground-Truth-Selective-PVC: 54
ProtoSSL-PVC: 217
ProtoECGNet-PVC: 219
AND-Criteria-PVC: 47


In [5]:
def get_case_sim_and_chunk_for_model(model_path: Path) -> dict[str, dict[str, np.ndarray]]:
    label_case_meta = dict()
    sims = np.load(model_path / "compute-embeddings/latest/test_embeds.npy")
    chunks = np.load(model_path / "compute-embeddings/latest/test_chunks.npy")
    proj = pd.read_csv(model_path / "project-prototypes-supervised/latest/projection_metadata.csv")

    for l in labels:
        label_idx = PTBXL_TARGETS.index(l)
        label_prototype_idxs = np.arange(prototypes_per_label) + label_idx * prototypes_per_label
        case_idxs = label_cases[l] # (CPL,)

        # label's prototype similarities
        case_label_sims = sims[np.ix_(case_idxs, label_prototype_idxs)] # (CPL, PPL)

        # label's prototypes with maximal similarity
        case_label_proto_idxs = case_label_sims.argmax(axis=1) # (CPL,)
        case_label_proto_sims = case_label_sims.max(axis=1) # (CPL,)
        global_proto_idxs = case_label_proto_idxs + label_idx * prototypes_per_label

        # case's chunk which resulted in those max similarities
        case_label_proto_chunks = chunks[case_idxs, global_proto_idxs] # (CPL,)

        label_case_meta[l] = {
            "similarity": case_label_proto_sims,
            "case_ecg_id": test_ds.sample_ids[case_idxs].numpy(),
            "case_ecg_window": case_label_proto_chunks,
            "global_proto_idxs": global_proto_idxs,
            "prototype_ecg_id": proj.loc[global_proto_idxs, "ecg_id"].to_numpy(),
            "prototype_ecg_window": proj.loc[global_proto_idxs, "chunk_idx"].to_numpy(),
        }

    return label_case_meta

In [6]:
protossl_meta = get_case_sim_and_chunk_for_model(protossl)
protoecgnet_meta = get_case_sim_and_chunk_for_model(protoecgnet)

# this shouldn't differ but just double check
# NOTE the maximally similar windows will differ given the different prototypes!
assert (protossl_meta["AMI"]["case_ecg_id"] == protoecgnet_meta["AMI"]["case_ecg_id"]).all()

In [7]:
LEAD_ORDER = [
    "I",
    "II",
    "III",
    "aVR",
    "aVL",
    "aVF",
    "V1",
    "V2",
    "V3",
    "V4",
    "V5",
    "V6",
]  # order in data
LEAD_TO_IDX = {name: i for i, name in enumerate(LEAD_ORDER)}
LEAD_LAYOUT = [
    ["I", "aVR", "V1", "V4"],
    ["II", "aVL", "V2", "V5"],
    ["III", "aVF", "V3", "V6"],
]
RHYTHM_LEADS = [LEAD_TO_IDX["II"]]
MM_PER_MV = 10
MM_PER_SEC = 25
PAPER_SPEED = MM_PER_SEC
GAIN = MM_PER_MV
N_FULL_ROWS = len(LEAD_LAYOUT) + len(RHYTHM_LEADS)
ROW_SPACING = 50
SAMPLING_RATE = 100  # Hz
DURATION = 10  # seconds

PRINTOUT_WIDTH_IN = (DURATION * PAPER_SPEED) / 25.4
PRINTOUT_HEIGHT_IN = (N_FULL_ROWS * ROW_SPACING) / 25.4

SEG_HEIGHT_IN = 1.4

FULL_SEG_VGAP_REL = 0.1 # relative to PRINTOUT_HEIGHT
LEFT_RIGHT_HGAP_REL = 0.15 # relative to PRINTOUT_WIDTH

# need to unnormalize waveforms
stds = np.asarray(PTBXL_CLIPPED_STDS)[:, None]
means = np.asarray(PTBXL_CLIPPED_MEANS)[:, None]


def plot_compare(
    *,
    left_ecg_id,
    left_ecg_window,
    right_ecg_id,
    right_ecg_window,
    sup_title: str | None = None,
    left_title: str | None = None,
    right_title: str | None = None,
    annotation: str | None = None,
):
    # make grid layout like:
    # [ecg_width][0.1*ecg_width][ecg_width] < full ECG plot
    # [...........0.1*ecg_height..........]
    # [ecg_width][0.1*ecg_width][ecg_width] < segment ECG plot, subplots of 3 rows, 4 cols, height=1.4in

    fig_width_in = PRINTOUT_WIDTH_IN * (2+LEFT_RIGHT_HGAP_REL)
    fig_height_in = PRINTOUT_HEIGHT_IN * (1+FULL_SEG_VGAP_REL) + SEG_HEIGHT_IN*3
    fig = plt.figure(figsize=(fig_width_in, fig_height_in))
    gs = fig.add_gridspec(
        nrows=5,
        ncols=9,
        width_ratios=[0.25]*4+[LEFT_RIGHT_HGAP_REL]+[0.25]*4,
        height_ratios=[PRINTOUT_HEIGHT_IN, FULL_SEG_VGAP_REL*PRINTOUT_HEIGHT_IN]+[SEG_HEIGHT_IN]*3,
    )

    ax_full_left = fig.add_subplot(gs[0, 0:4])
    ax_full_right = fig.add_subplot(gs[0, 5:9])
    axs_seg_left: list[list] = [[None for j in range(4)] for i in range(3)]
    axs_seg_right: list[list] = [[None for j in range(4)] for i in range(3)]
    for axs_seg, col_offset in [
        (axs_seg_left, 0),
        (axs_seg_right, 5),
    ]:
        for i in range(3):
            for j in range(4):
                ax = fig.add_subplot(gs[2+i, j+col_offset])
                if i != 0 or j != 0:
                    ax.sharex(axs_seg[0][0])
                axs_seg[i][j] = ax

    axs_seg_left = np.asarray(axs_seg_left)
    axs_seg_right = np.asarray(axs_seg_right)

    plot_2d_prototype(left_ecg_id, left_ecg_window, ax_full_left, axs_seg_left)
    plot_2d_prototype(right_ecg_id, right_ecg_window, ax_full_right, axs_seg_right)

    if left_title is not None:
        ax_full_left.set_title(left_title, fontsize=32, color="tab:blue", fontweight="heavy")
    if right_title is not None:
        ax_full_right.set_title(right_title, fontsize=32, color="tab:blue", fontweight="heavy")
    if annotation is not None:
        fig.text(
            x=0.507,   # horizontal center of figure
            y=0.5,   # vertical center of figure
            s=annotation,
            ha="center",  # horizontal alignment
            va="center",  # vertical alignment
            fontsize=16,
            color="tab:blue",
            fontweight="heavy",
        )

    if sup_title is not None:
        fig.text(x=0.5, y=0.955, s=sup_title, fontsize=32, color="tab:blue", fontweight="heavy", ha="center", va="bottom")
    fig.tight_layout()
    return fig


def plot_2d_prototype(ecg_id, ecg_window, full_ax: plt.Axes, seg_axs: np.ndarray):
    # Prepare ECG signal (not exactly equivalent to raw ECG given unnormalization)
    if ecg_id in ecg_id_to_idx:
        ecg_signal = test_ds[ecg_id_to_idx[ecg_id]]["waveform"].numpy()
    else:
        # dumb heuristic to check if ecg comes from test (the test ECGs) or from train (the projection ECGs)
        ecg_signal = train_ds[train_ecg_id_to_idx[ecg_id]]["waveform"].numpy()

    assert ecg_signal.shape == (
        12,
        1000,
    ), f"Expected shape (12,1000), got {ecg_signal.shape}"
    ecg_signal = (ecg_signal * stds) + means
    ecg_signal = remove_baseline_wander(ecg_signal)

    # --- Full ECG with highlight ---
    plot_full(full_signal=ecg_signal, ax=full_ax)

    # assumes 10 second recording at 100 Hz with 1 second windows and 50% overlap
    highlight_start = ecg_window / 2  # e.g. window 11 --> starts at 5.5 sec
    highlight_end = highlight_start + 1
    full_ax.axvspan(highlight_start, highlight_end, color="blue", alpha=0.2, zorder=1)
    ymin, ymax = full_ax.get_ylim()

    # --- Segment Cutout (print layout) ---
    raw_start, raw_end = int(highlight_start * SAMPLING_RATE), int(highlight_end * SAMPLING_RATE)
    segment_signal = ecg_signal[:, raw_start:raw_end]
    plot_segment(segment_signal=segment_signal, axs=seg_axs)

    callout_left = patches.ConnectionPatch(
        xyA=(highlight_start, ymin), # bottom left of highlight window
        coordsA="data",
        xyB=(0, 1), # top left corner of top left segment sub-ax
        coordsB="axes fraction",
        axesA=full_ax,
        axesB=seg_axs[0, 0],
        color="black",
        linewidth=1.5,
    )
    callout_right = patches.ConnectionPatch(
        xyA=(highlight_end, ymin), # bottom right of highlight window
        coordsA="data",
        xyB=(1, 1), # top right corner of top right segment sub-ax
        coordsB="axes fraction",
        axesA=full_ax,
        axesB=seg_axs[0, -1],
        color="black",
        linewidth=1.5,
    )

    fig = full_ax.get_figure()
    fig.add_artist(callout_left)
    fig.add_artist(callout_right)


def plot_full(full_signal: np.ndarray, ax: plt.Axes):
    assert full_signal.shape == (12, DURATION * SAMPLING_RATE)
    full_signal = full_signal.T  # reshape to (T_steps, L)

    ax.set_xlim(0, DURATION)
    ax.set_ylim(0, N_FULL_ROWS * ROW_SPACING)
    ax.axis("off")

    # draw paper grid
    for x in np.arange(0, DURATION + 0.04, 0.04):
        ax.axvline(x, color="pink", linewidth=0.5, zorder=0)
    for y in np.arange(0, N_FULL_ROWS * ROW_SPACING + 0.1, 1):
        is_large = (y * 0.1) % 0.5 == 0
        ax.axhline(
            y,
            color="pink" if not is_large else "red",
            linewidth=0.5 if not is_large else 1.0,
            zorder=0,
        )
    for x in np.arange(0, DURATION + 0.2, 0.2):
        ax.axvline(x, color="red", linewidth=1.0, zorder=0)

    label_fontsize = 12
    row_baseline = N_FULL_ROWS * ROW_SPACING - ROW_SPACING / 2

    for row_idx, lead_row in enumerate(LEAD_LAYOUT):
        for col_idx, lead in enumerate(lead_row):
            lead_idx = LEAD_TO_IDX[lead]
            start = col_idx * 250
            end = (col_idx + 1) * 250
            signal = full_signal[start:end, lead_idx] * GAIN
            t = np.linspace(col_idx * 2.5, (col_idx + 1) * 2.5, 250)
            v_offset = row_baseline - row_idx * ROW_SPACING
            ax.plot(t, signal + v_offset, color="black", linewidth=1.0)
            ax.text(
                t[0] + 0.1,
                v_offset + 16,
                lead,
                fontsize=label_fontsize,
                fontweight="bold",
            )

    # Rhythm leads: full 10 seconds
    t = np.linspace(0, 10, full_signal.shape[0])
    for i, lead_idx in enumerate(RHYTHM_LEADS):
        v_offset = row_baseline - (len(LEAD_LAYOUT) + i) * ROW_SPACING
        signal = full_signal[:, lead_idx] * GAIN
        ax.plot(t, signal + v_offset, color="black", linewidth=1.0)
        ax.text(
            0.1,
            v_offset + 16,
            f"{LEAD_ORDER[lead_idx]}",
            fontsize=label_fontsize,
            fontweight="bold",
        )


def plot_segment(segment_signal: np.ndarray, axs: np.ndarray):
    assert axs.shape == (3, 4)  # to match layout
    assert segment_signal.shape[0] == 12  # signal should be (L, T_steps)
    t_steps = segment_signal.shape[1]
    t_segment = np.linspace(0, t_steps / SAMPLING_RATE, t_steps)

    for row in range(3):
        for col in range(4):
            lead = LEAD_LAYOUT[row][col]
            i = LEAD_TO_IDX[lead]
            ax: plt.Axes = axs[row, col]
            ax.plot(t_segment, segment_signal[i], color="black", linewidth=1.0)
            ax.set_xlim(0, t_segment[-1])
            ax.set_ylim(
                np.min(segment_signal[i]) - 0.5, np.max(segment_signal[i]) + 0.5
            )

            # Draw lead label to the left of the plot
            ax.text(
                x=-0.1,
                y=0.8,
                s=lead,
                fontsize=10,
                fontweight="bold",
                transform=ax.transAxes,
                ha="right",
                va="bottom",
            )

            # ECG grid
            for x in np.arange(t_segment[0], t_segment[-1] + 0.04, 0.04):
                ax.axvline(x, color="pink", linewidth=0.3, alpha=0.5)
            for x in np.arange(t_segment[0], t_segment[-1], 0.2):
                ax.axvline(x, color="red", linewidth=0.5, alpha=0.7)
            ax.axhline(0, color="red", linewidth=0.5, alpha=0.5)

            ax.tick_params(left=False, labelleft=False, bottom=False, labelbottom=False)


def remove_baseline_wander(X, cutoff=0.5, order=1):
    """
    Applies a high-pass Butterworth filter to remove baseline wander.
    No need to apply a low-pass filter if using 100 Hz data
    """
    assert X.ndim == 2 and X.shape == (12, 1000)
    X = X.copy()

    b, a = butter(order, cutoff / (SAMPLING_RATE / 2), btype='high', analog=False)
    for j in range(12):
        X[j, :] = filtfilt(b, a, X[j, :])

    return X

In [8]:
protossl_a_or_b = rng.choice(2, len(labels) * cases_per_label)
do_plot = True
unblind = True

study_idxs = [(i, j, label_name) for i, label_name in enumerate(labels) for j in range(cases_per_label)]
metadata = []
for (_i, j, label_name) in tqdm(study_idxs):
    # i is the label num
    # j is the case num for the label
    study_idx = _i * cases_per_label + j + 1 # index from 1 in survey designer
    if protossl_a_or_b[study_idx - 1] == 0:
        a_meta = protossl_meta
        b_meta = protoecgnet_meta
    else:
        a_meta = protoecgnet_meta
        b_meta = protossl_meta

    metadata.append({
        "Study Index": study_idx,
        "Label": label_name,
        "Case ECG ID": protossl_meta[label_name]["case_ecg_id"][j],
        "ProtoSSL Assignment": "A" if protossl_a_or_b[study_idx - 1] == 0 else "B",
        "ProtoSSL Proto Index": protossl_meta[label_name]["global_proto_idxs"][j],
        "ProtoSSL Proto ECG ID": protossl_meta[label_name]["prototype_ecg_id"][j],
        "ProtoSSL Proto ECG Window": protossl_meta[label_name]["prototype_ecg_window"][j],
        "ProtoSSL Case ECG Window": protossl_meta[label_name]["case_ecg_window"][j],
        "ProtoSSL Case Similarity": protossl_meta[label_name]["similarity"][j],
        "ProtoECGNet Assignment": "A" if protossl_a_or_b[study_idx - 1] == 1 else "B",
        "ProtoECGNet Proto Index": protoecgnet_meta[label_name]["global_proto_idxs"][j],
        "ProtoECGNet Proto ECG ID": protoecgnet_meta[label_name]["prototype_ecg_id"][j],
        "ProtoECGNet Proto ECG Window": protoecgnet_meta[label_name]["prototype_ecg_window"][j],
        "ProtoECGNet Case ECG Window": protoecgnet_meta[label_name]["case_ecg_window"][j],
        "ProtoECGNet Case Similarity": protoecgnet_meta[label_name]["similarity"][j],
    })

    if not do_plot:
        continue

    model_a_name = "Model A"
    model_b_name = "Model B"
    if unblind:
        model_a_name = "ProtoSSL HEEDB" if protossl_a_or_b[study_idx - 1] == 0 else "SupProto Direct"
        model_b_name = "ProtoSSL HEEDB" if protossl_a_or_b[study_idx - 1] == 1 else "SupProto Direct"

    label_clean = label_name
    if label_name.endswith("BBB"):
        label_clean = label_name[1:]

    global_fig = plot_compare(
        sup_title=f"Case {study_idx}: {label_clean}",
        # case on left
        left_ecg_id=a_meta[label_name]["prototype_ecg_id"][j],
        left_ecg_window=a_meta[label_name]["prototype_ecg_window"][j],
        left_title=model_a_name,
        # model on right
        right_ecg_id=b_meta[label_name]["prototype_ecg_id"][j],
        right_ecg_window=b_meta[label_name]["prototype_ecg_window"][j],
        right_title=model_b_name,
    )
    global_fig.savefig(f"images/{study_idx:02d}_global_proto.png", dpi=300)
    a_fig = plot_compare(
        sup_title=f"Case {study_idx}: {label_clean}",
        # case on left
        left_ecg_id=a_meta[label_name]["case_ecg_id"][j],
        left_ecg_window=a_meta[label_name]["case_ecg_window"][j],
        left_title=f"Test",
        # model on right
        right_ecg_id=a_meta[label_name]["prototype_ecg_id"][j],
        right_ecg_window=a_meta[label_name]["prototype_ecg_window"][j],
        right_title=model_a_name,
        # annotate similarity
        annotation=f"{a_meta[label_name]['similarity'][j]:0.3f} Similarity",
    )
    a_fig.savefig(f"images/{study_idx:02d}_case_proto_A.png", dpi=300)
    b_fig = plot_compare(
        sup_title=f"Case {study_idx}: {label_clean}",
        # case on left
        left_ecg_id=b_meta[label_name]["case_ecg_id"][j],
        left_ecg_window=b_meta[label_name]["case_ecg_window"][j],
        left_title=f"Test",
        # model on right
        right_ecg_id=b_meta[label_name]["prototype_ecg_id"][j],
        right_ecg_window=b_meta[label_name]["prototype_ecg_window"][j],
        right_title=model_b_name,
        # annotate similarity
        annotation=f"{b_meta[label_name]['similarity'][j]:0.3f} Similarity",
    )
    b_fig.savefig(f"images/{study_idx:02d}_case_proto_B.png", dpi=300)

    plt.close("all")

metadata = pd.DataFrame.from_dict(metadata, orient="columns")
# metadata.to_csv("metadata.csv", index=False)

100%|██████████| 20/20 [02:10<00:00,  6.54s/it]


In [9]:
old = pd.read_csv("metadata.csv", dtype={
    "ProtoSSL Case Similarity": np.float32,
    "ProtoECGNet Case Similarity": np.float32,
})

assert old.equals(metadata)

In [10]:
metadata

,Study Index,Label,Case ECG ID,ProtoSSL Assignment,ProtoSSL Proto Index,ProtoSSL Proto ECG ID,ProtoSSL Proto ECG Window,ProtoSSL Case ECG Window,ProtoSSL Case Similarity,ProtoECGNet Assignment,ProtoECGNet Proto Index,ProtoECGNet Proto ECG ID,ProtoECGNet Proto ECG Window,ProtoECGNet Case ECG Window,ProtoECGNet Case Similarity
0,1,AMI,3875,A,255,21716,0,4,0.774049,B,265,2484,5,15,0.889128
1,2,AMI,11319,A,259,12462,14,18,0.756195,B,256,1366,9,10,0.880022
2,3,AMI,20240,B,255,21716,0,17,0.775543,A,260,15501,1,17,0.891802
3,4,AMI,12235,A,255,21716,0,1,0.720044,B,257,11158,5,18,0.715555
4,5,AMI,20313,B,259,12462,14,6,0.741223,A,258,12462,6,16,0.726461
5,6,CRBBB,8712,B,207,2894,8,0,0.770370,A,209,14666,14,6,0.769873
6,7,CRBBB,20252,B,200,12902,4,8,0.755400,A,203,21212,14,10,0.776256
7,8,CRBBB,19134,B,208,8994,13,18,0.780665,A,203,21212,14,12,0.870419
8,9,CRBBB,8041,A,206,14371,15,11,0.806429,B,197,10938,4,12,0.792908
9,10,CRBBB,20674,B,208,8994,13,1,0.741794,A,205,9216,3,3,0.824757


In [11]:
# check how many times duplicate global prototype comparisons are made (acceptable if only 1 or 2 dupes)
metadata[metadata[["ProtoSSL Proto Index", "ProtoECGNet Proto Index"]].duplicated(keep=False)]

,Study Index,Label,Case ECG ID,ProtoSSL Assignment,ProtoSSL Proto Index,ProtoSSL Proto ECG ID,ProtoSSL Proto ECG Window,ProtoSSL Case ECG Window,ProtoSSL Case Similarity,ProtoECGNet Assignment,ProtoECGNet Proto Index,ProtoECGNet Proto ECG ID,ProtoECGNet Proto ECG Window,ProtoECGNet Case ECG Window,ProtoECGNet Case Similarity
15,16,PVC,6180,B,635,11869,2,1,0.770428,A,631,13958,4,10,0.965846
17,18,PVC,12576,B,635,11869,2,5,0.735454,A,631,13958,4,1,0.788782
18,19,PVC,10240,A,630,1633,3,2,0.803970,B,638,632,12,11,0.875438
19,20,PVC,16994,B,630,1633,3,7,0.772867,A,638,632,12,6,0.948079


In [12]:
# final check to make sure the ones we've fixed are where they're supposed to be
for i, (label_name, ecg_ids) in enumerate(fixed_cases.items()):
    for j, ecg_id in enumerate(ecg_ids):
        if ecg_id is None:
            continue
        assert metadata.loc[i * cases_per_label + j, "Case ECG ID"] == ecg_id

# final check to make sure we're not using any of the ones we've excluded
all_excluded = [ecg_id for ecg_ids in exclude_cases.values() for ecg_id in ecg_ids]
assert not metadata["Case ECG ID"].isin(all_excluded).any()